# 03 · Recomendação no mapa-múndi
    Execute os notebooks 01 e 02 para publicar as observações e as previsões.
    O perfil da base é fixo. O mês seleciona somente os valores terrestres.
    O cluster aparece no mês usado para ajustar o agrupamento. As coordenadas vêm do de-para.

## Configurar caminhos

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/feynmans_legacy/analog_study.py").exists()
)
sys.path.insert(0, str(ROOT / 'src'))
TERRA = ROOT / 'data/terra'
RUN = TERRA / 'analise_analogos'
RUN.mkdir(parents=True, exist_ok=True)

## Escolher o mês, a base e o nicho

In [2]:
import plotly.express as px

from feynmans_legacy.analog_study import MIN_COVERAGE, profile_criteria, recommend

ANOMES = None  # None usa o ultimo mes observado
CORPO = "lua"
BASE = None  # None usa a primeira base da spec selecionada
NICHO = "Todos"
# Fração mínima do peso do perfil com dados no local. Critérios sem dado saem da nota
# e os pesos restantes são renormalizados. 1.0 exige todos os critérios.
COBERTURA_MINIMA = MIN_COVERAGE


## Buscar os dados do mês

In [3]:
import json

selection = json.loads((RUN / "selecao.json").read_text(encoding="utf-8"))
latest = max(selection["anomes_observados"])
ANOMES = latest if ANOMES is None else ANOMES
if ANOMES <= latest:
    origem = "observada"
    values = pd.read_parquet(
        TERRA / "terra.parquet", filters=[("anomes", "==", ANOMES)]
    ).drop(columns="anomes")
    if values.empty:
        raise ValueError("Sem observações para o anomes selecionado.")
else:
    origem = "prevista"
    future = pd.read_parquet(
        RUN / "previsoes.parquet", filters=[("anomes", "==", ANOMES)]
    )
    if future.empty:
        raise ValueError("Sem previsões publicadas para o anomes selecionado.")
    values = future.pivot(  # noqa: PD010
        index="location_id", columns="variavel", values="valor_previsto"
    ).reset_index()
values = values.merge(
    pd.read_parquet(RUN / "estaticas.parquet"),
    on="location_id", how="left", validate="one_to_one",
)

## Aplicar o perfil e ordenar os locais

In [4]:
reference = pd.read_parquet(ROOT / "data" / CORPO / f"{CORPO}.parquet")
if BASE is None:
    if reference.empty:
        raise ValueError(f"Sem bases cadastradas na spec de {CORPO}.")
    BASE = reference["base_id"].iloc[0]
criteria = profile_criteria(ROOT / "config/nichos.json", CORPO, NICHO, reference, BASE)
ranking = recommend(
    values, pd.read_parquet(RUN / "coordenadas.parquet"), criteria,
    min_coverage=COBERTURA_MINIMA,
)
if ranking.empty:
    raise ValueError("Nenhum local com cobertura mínima e coordenadas para essa consulta.")


## Adicionar os clusters do mês

In [5]:
clusters = pd.read_parquet(RUN / "clusters.parquet").query("anomes == @ANOMES")
ranking = ranking.merge(clusters[["location_id", "cluster"]], on="location_id", how="left",
                        validate="one_to_one")
ranking["recomendado"] = ["Recomendado"] + ["Candidato"] * (len(ranking) - 1)


## Conferir a cobertura do ranking

In [6]:
print(f"{origem.capitalize()} | {len(ranking):,} locais elegíveis de {len(values):,} consultados")
completos = ranking["cobertura"].ge(1 - 1e-9).sum()
print(f"{completos:,} com todos os critérios; {len(ranking) - completos:,} com cobertura parcial")

Observada | 77,543 locais elegíveis de 85,726 consultados
58,283 com todos os critérios; 19,260 com cobertura parcial


## Exibir os locais no mapa-múndi

In [17]:
LIMITE_MAPA = 100
points = ranking.sort_values("nota", ascending=False).head(LIMITE_MAPA).copy()
points["cluster"] = points["cluster"].astype("string").fillna("Indisponível")
figure = px.scatter_geo(
    points, lat="latitude", lon="longitude", color="recomendado", hover_name="location_id",
    hover_data=[
        "nota", "cobertura", "cluster",
        *criteria, *[f"alvo_{variable}" for variable in criteria],
    ],
    projection="natural earth",
    title=f"{BASE} · {NICHO} · {ANOMES} ({origem}) · {len(points):,} de {len(ranking):,} locais",
)
figure.update_traces(
    name=f"Próximos no ranking ({len(points) - 1:,})", marker_size=5,
    selector={"name": "Candidato"},
)
figure.update_traces(marker_size=11, selector={"name": "Recomendado"})

figure.show()

## Ver os locais mais recomendados

In [10]:
from IPython.display import display

LIMITE_TABELA = 1000
colunas = {
    "location_id": "Local",
    "nota": "Nota de correspondência (0–100)",
    "cobertura": "Cobertura (%)",
    "latitude": "Latitude",
    "longitude": "Longitude",
}
recomendados = ranking.head(LIMITE_TABELA)[list(colunas)].reset_index(drop=True)
recomendados.index += 1
recomendados.index.name = "Posição"
recomendados.insert(0, "Base de referência", BASE)
recomendados.insert(0, "Corpo de referência", {"lua": "Lua", "marte": "Marte"}[CORPO])
recomendados["cobertura"] = recomendados["cobertura"] * 100

display(
    recomendados.round({"nota": 2, "cobertura": 1, "latitude": 3, "longitude": 3})
    .rename(columns=colunas)
)

,Corpo de referência,Base de referência,Local,Nota de correspondência (0–100),Cobertura (%),Latitude,Longitude
Posição,,,,,,,
1,Lua,LUA_POLO_SUL,T072256,76.52,76.2,-79.75,159.75
2,Lua,LUA_POLO_SUL,T078504,74.56,76.2,-85.25,-177.75
3,Lua,LUA_POLO_SUL,T075996,74.43,76.2,-83.25,-48.25
4,Lua,LUA_POLO_SUL,T076662,74.29,76.2,-83.75,-50.75
5,Lua,LUA_POLO_SUL,T077329,74.03,76.2,-84.25,-54.75
...,...,...,...,...,...,...,...
996,Lua,LUA_POLO_SUL,T075954,64.97,76.2,-83.25,-71.75
997,Lua,LUA_POLO_SUL,T000335,64.96,90.5,81.75,58.25
998,Lua,LUA_POLO_SUL,T075029,64.96,76.2,-82.25,106.75


## Melhor opção por base lunar e marciana

Compara as bases no mês selecionado e usa **Todos** como nicho padrão. Cada linha apresenta o local terrestre mais bem pontuado para aquela base.

In [12]:
NICHO_COMPARACAO = "Todos"  # use NICHO para comparar com o nicho da consulta atual

config_path = ROOT / "config" / "nichos.json"
nichos_por_corpo = json.loads(config_path.read_text(encoding="utf-8"))
coordenadas_comparacao = pd.read_parquet(RUN / "coordenadas.parquet")
melhores_por_base = []

for corpo_base, nome_corpo in (("lua", "Lua"), ("marte", "Marte")):
    perfis = pd.read_parquet(ROOT / "data" / corpo_base / f"{corpo_base}.parquet")

    for base_id in perfis["base_id"]:
        linha = {
            "Corpo de referência": nome_corpo,
            "Base de referência": base_id,
            "Local recomendado": None,
            "Nota de correspondência (0–100)": None,
            "Cobertura (%)": None,
            "Latitude": None,
            "Longitude": None,
            "Status": "Disponível",
        }

        if NICHO_COMPARACAO != "Todos" and NICHO_COMPARACAO not in nichos_por_corpo[corpo_base]:
            linha["Status"] = f"Nicho indisponível para {nome_corpo}"
        else:
            criterios_base = profile_criteria(
                config_path, corpo_base, NICHO_COMPARACAO, perfis, base_id
            )
            faltantes = sorted(set(criterios_base).difference(values.columns))

            if faltantes:
                linha["Status"] = f"Critérios sem dados no mês: {', '.join(faltantes)}"
            else:
                ranking_base = recommend(
                    values, coordenadas_comparacao, criterios_base,
                    min_coverage=COBERTURA_MINIMA,
                )
                if ranking_base.empty:
                    linha["Status"] = "Nenhum local com a cobertura mínima"
                else:
                    melhor = ranking_base.iloc[0]
                    linha.update({
                        "Local recomendado": melhor["location_id"],
                        "Nota de correspondência (0–100)": round(float(melhor["nota"]), 2),
                        "Cobertura (%)": round(float(melhor["cobertura"]) * 100, 1),
                        "Latitude": round(float(melhor["latitude"]), 3),
                        "Longitude": round(float(melhor["longitude"]), 3),
                    })

        melhores_por_base.append(linha)

print(f"{ANOMES} · {NICHO_COMPARACAO} · dados de origem {origem}")
tabela_bases = pd.DataFrame(melhores_por_base)
display(tabela_bases)

pontos_bases = tabela_bases.loc[tabela_bases["Status"].eq("Disponível")]
if pontos_bases.empty:
    print("Sem recomendações para exibir no mapa nesta seleção.")
else:
    mapa_bases = px.scatter_geo(
        pontos_bases,
        lat="Latitude",
        lon="Longitude",
        color="Corpo de referência",
        text="Base de referência",
        hover_name="Base de referência",
        hover_data={
            "Local recomendado": True,
            "Nota de correspondência (0–100)": ":.2f",
            "Cobertura (%)": ":.1f",
        },
        color_discrete_map={"Lua": "#2563eb", "Marte": "#c2410c"},
        projection="natural earth",
        title=f"Melhor local na Terra por base · {ANOMES} · {NICHO_COMPARACAO}",
    )
    mapa_bases.update_traces(
        textposition="top center",
        marker_size=10,
        marker_line_width=1,
        marker_line_color="white",
    )
    mapa_bases.show()


202608 · Todos · dados de origem observada


,Corpo de referência,Base de referência,Local recomendado,Nota de correspondência (0–100),Cobertura (%),Latitude,Longitude,Status
0,Lua,LUA_POLO_SUL,T072256,76.52,76.2,-79.75,159.75,Disponível
1,Lua,LUA_MARE_TUBOS,T039384,89.02,100.0,25.75,-4.75,Disponível
2,Marte,MARTE_JEZERO,T042068,83.68,100.0,20.75,33.25,Disponível
3,Marte,MARTE_GALE,T034227,82.04,100.0,34.25,-115.25,Disponível
4,Marte,MARTE_MERIDIANI,T031402,78.66,100.0,39.25,82.25,Disponível
5,Marte,MARTE_GELO_MEDIAS_LAT,T070922,87.84,76.7,-78.75,44.25,Disponível
6,Marte,MARTE_THARSIS_TUBOS,T057810,85.40,100.0,-25.25,-68.25,Disponível
7,Marte,MARTE_MEDUSAE_YARDANGS,T036531,79.59,100.0,30.75,59.75,Disponível
